# ORAN-Control Demo

This notebook demonstrates operator-driven Open RAN KPM generation.

You only specify:

- scheduler
- PRB allocation
- offered traffic
- active flows
- desired throughput
- desired buffer

Model-specific parameters, latent context, guidance strength,
verifier thresholds, and retry logic are handled internally.


## 1. Load the generator


In [ ]:
from oran_control import (
    ORANGenerator,
    NetworkCondition,
    NetworkRequest,
)

generator = ORANGenerator.from_pretrained()

print("Generator ready.")


## 2. Define the Open RAN operating condition


In [ ]:
condition = (
    NetworkCondition()
    .use_scheduler("PF")
    .allocate_prbs(
        embb=30,
        urllc=20,
    )
    .set_traffic(
        embb_mbps=12.0,
        urllc_mbps=0.04,
        embb_flows=3,
        urllc_flows=2,
    )
)

print(condition.summary())


## 3. Define the desired network behavior


In [ ]:
request = (
    NetworkRequest()
    .require_embb_throughput(
        10.0
    )
    .limit_embb_buffer(
        50.0
    )
)

print(request.summary())


## 4. Generate and verify a 60-second KPM trace


In [ ]:
result = generator.generate(
    condition=condition,
    request=request,
    seed=2026,
)

print(result.report)


## 5. Inspect the generated KPM trace


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

trace = np.asarray(result.trace)

time_s = np.arange(
    len(trace)
)

print("Trace shape:", trace.shape)


### eMBB throughput


In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(
    time_s,
    trace[:, 0],
)
plt.axhline(
    10.0,
    linestyle="--",
    label="Requested minimum",
)
plt.xlabel("Time [s]")
plt.ylabel("eMBB throughput [Mbps]")
plt.title("Generated eMBB Throughput")
plt.legend()
plt.show()


### eMBB buffer


In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(
    time_s,
    trace[:, 1] / 1000.0,
)
plt.axhline(
    50.0,
    linestyle="--",
    label="Requested maximum",
)
plt.xlabel("Time [s]")
plt.ylabel("eMBB buffer [KB]")
plt.title("Generated eMBB Buffer")
plt.legend()
plt.show()


## 6. Save the generated trace


In [ ]:
output_path = result.save(
    "generated_trace.csv"
)

print("Saved:", output_path)
